# WasteValue AI — เทรนโมเดลและประเมินผล (โจทย์ข้อ 3–4)

notebook นี้อ่านไฟล์จากปาล์ม (`split.csv`, `class_weights.json`, `category_map.json`) แล้วทำตาม **Model Evaluation** ที่เรียน:

1. **5-fold cross validation** บนส่วน train เปรียบเทียบ 2 การทดลอง
   - **การทดลอง A:** ไม่ใช้ data augmentation
   - **การทดลอง B:** ใช้ data augmentation (หมุน พลิก ครอบตัด ปรับแสง)
2. เลือกการทดลองที่ **macro F1 เฉลี่ย** สูงกว่า → เทรนใหม่กับ train ทั้งหมด
3. วัดผลกับ **test ครั้งเดียว** ทั้ง 2 ระดับ: ระดับคลาส (13 คลาส) และระดับประเภทขยะ (5 ประเภท)
4. export `model.pt` ให้ทำเว็บ

**โมเดล:** CNN แบบ Transfer Learning ใช้ MobileNetV2 ที่เทรนมาแล้วบน ImageNet แช่แข็งชั้น convolution ไว้ แล้วฝึกเฉพาะชั้นสุดท้ายให้แยก 13 คลาส

**ก่อนรัน:** เมนู Runtime → Change runtime type → **T4 GPU**
ผลแต่ละ fold ถูกเซฟลง Drive ทันที ถ้า Colab หลุดกลางทาง กด Run all ใหม่ได้ ระบบจะข้าม fold ที่ทำเสร็จแล้ว

## ตั้งค่า

In [ ]:
DRIVE_FOLDER = "/content/drive/MyDrive/WasteValue"   # โฟลเดอร์ทีม (ปาล์มแชร์มา)
ZIP_NAME     = "garbage-classification.zip"
EXTRACT_TO   = "/content/data"
PALM_OUT     = DRIVE_FOLDER + "/palm_output"           # ไฟล์จากปาล์ม
OUT_DIR      = DRIVE_FOLDER + "/fanta_output"          # ผลลัพธ์ของ notebook นี้

MODE = "cv"          # "cv" = 5-fold ตามที่เรียน (~1 ชม.) | "quick" = ใช้ fold 0 เป็น validation อย่างเดียว (~15 นาที)
EPOCHS = 6           # จำนวนรอบต่อการเทรน 1 ครั้ง
BATCH_SIZE = 64
LR = 1e-3
USE_CLASS_WEIGHTS = True   # ให้น้ำหนักคลาสที่มีภาพน้อยมากขึ้น
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ต้องตรงกับเว็บ
SEED = 42

In [ ]:
import os, json, glob, time, copy, zipfile, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split, StratifiedKFold

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("ไม่ได้รันบน Colab: ข้ามการเชื่อม Google Drive")
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("⚠️ ไม่ได้ใช้ GPU จะช้ามาก ไปที่ Runtime → Change runtime type → T4 GPU")
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)

## ขั้นที่ 1: โหลดข้อมูล
แตก zip ไฟล์เดียวกับที่ปาล์มใช้ แล้วอ่าน `split.csv`

**ถ้าปาล์มยังไม่เสร็จ** notebook จะสร้าง split ชั่วคราวจากข้อมูลดิบ (คลาสละ 60 ภาพ) ให้ลองรันทั้ง notebook ดูว่าโค้ดผ่านก่อน ผลที่ได้จากโหมดนี้**ห้ามเอาไปใช้ในรายงาน**

In [ ]:
zip_path = os.path.join(DRIVE_FOLDER, ZIP_NAME)
if not (os.path.isdir(EXTRACT_TO) and os.listdir(EXTRACT_TO)):
    assert os.path.exists(zip_path), f"ไม่เจอ {zip_path} ให้ปาล์มแชร์โฟลเดอร์ทีมมาก่อน"
    print("กำลังแตกไฟล์ (ประมาณ 1–2 นาที)...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(EXTRACT_TO)
DATA_ROOT = None
for root, dirs, _ in os.walk(EXTRACT_TO):
    if {"battery", "plastic", "paper"}.issubset(set(dirs)):
        DATA_ROOT = root
        break
assert DATA_ROOT, "หาโฟลเดอร์คลาสไม่เจอ"
print("DATA_ROOT =", DATA_ROOT)

split_path = os.path.join(PALM_OUT, "split.csv")
TEMP_MODE = not os.path.exists(split_path)
if not TEMP_MODE:
    split = pd.read_csv(split_path)
    CLASSES = json.load(open(os.path.join(PALM_OUT, "class_weights.json")))["classes"]
    CATEGORY_MAP = json.load(open(os.path.join(PALM_OUT, "category_map.json")))
    print(f"ใช้ split.csv จากปาล์ม: {len(split)} ภาพ, {len(CLASSES)} คลาส")
else:
    print("⚠️ ยังไม่มี split.csv จากปาล์ม: สร้าง split ชั่วคราวเพื่อทดสอบโค้ด (ห้ามใช้ผลนี้ในรายงาน)")
    rows = []
    for cls in sorted(os.listdir(DATA_ROOT)):
        d = os.path.join(DATA_ROOT, cls)
        if os.path.isdir(d):
            fs = sorted(os.listdir(d))[:60]
            rows += [{"path": f"{cls}/{f}", "label": cls} for f in fs]
    split = pd.DataFrame(rows)
    CLASSES = sorted(split["label"].unique())
    CATEGORY_MAP = {c: c for c in CLASSES}
    tr, te = train_test_split(split, test_size=0.15, stratify=split["label"], random_state=SEED)
    tr, te = tr.assign(split="train", fold=-1), te.assign(split="test", fold=-1)
    for k, (_, idx) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, tr["label"])):
        tr.iloc[idx, tr.columns.get_loc("fold")] = k
    split = pd.concat([tr, te]).reset_index(drop=True)
    split["label_idx"] = split["label"].map({c: i for i, c in enumerate(CLASSES)})
    split["category"] = split["label"]

CATEGORIES = sorted(set(CATEGORY_MAP.values()))
cls_to_cat = np.array([CATEGORIES.index(CATEGORY_MAP[c]) for c in CLASSES])
print("คลาส:", CLASSES)
print("ประเภทขยะ:", CATEGORIES)
print(split.groupby(["split"]).size().to_string())

## ขั้นที่ 2: โหลดภาพเข้าหน่วยความจำ
ย่อทุกภาพเป็น 256×256 แล้วเก็บไว้ใน RAM ครั้งเดียว ทำให้เทรนแต่ละรอบเร็วขึ้นมาก (ใช้เวลาโหลดประมาณ 2–4 นาที)

In [ ]:
CACHE_SIZE = 256
cache_file = f"/content/cache_{CACHE_SIZE}_{len(split)}.npy"
if os.path.exists(cache_file):
    images = np.load(cache_file)
else:
    images = np.zeros((len(split), CACHE_SIZE, CACHE_SIZE, 3), dtype=np.uint8)
    ok = np.ones(len(split), dtype=bool)
    t0 = time.time()
    for i, rel in enumerate(split["path"]):
        try:
            with Image.open(os.path.join(DATA_ROOT, rel)) as im:
                images[i] = np.asarray(im.convert("RGB").resize((CACHE_SIZE, CACHE_SIZE), Image.BILINEAR))
        except Exception:
            ok[i] = False                         # ภาพเสีย (เจอได้เฉพาะโหมดชั่วคราว เพราะปาล์มตัดออกแล้ว)
        if (i + 1) % 2000 == 0:
            print(f"โหลดแล้ว {i + 1}/{len(split)} ({time.time() - t0:.0f} วินาที)")
    if not ok.all():
        print(f"⚠️ เปิดไม่ได้ {int((~ok).sum())} ภาพ ข้ามไป")
        split, images = split[ok].reset_index(drop=True), images[ok]
        cache_file = f"/content/cache_{CACHE_SIZE}_{len(split)}.npy"
    try:
        np.save(cache_file, images)
    except OSError:
        pass
labels = split["label_idx"].to_numpy()
print("images:", images.shape, f"({images.nbytes / 1e9:.1f} GB)")

## ขั้นที่ 3: เตรียม transform และโมเดล
- **ไม่มี augmentation (A):** ย่อภาพเป็น 224×224 แล้ว normalize
- **มี augmentation (B):** สุ่มครอบตัด พลิกซ้ายขวา หมุน ±15° ปรับแสงและสี (ใช้เฉพาะตอนเทรน)
- validation และ test ใช้แบบไม่มี augmentation เสมอ

In [ ]:
tf_plain = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_aug = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
    transforms.ColorJitter(0.3, 0.3, 0.3),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
EXPERIMENTS = {"A_no_aug": tf_plain, "B_aug": tf_aug}

class CachedSet(Dataset):
    def __init__(self, idx, tf):
        self.idx, self.tf = np.asarray(idx), tf
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        return self.tf(Image.fromarray(images[j])), int(labels[j])

def make_loader(idx, tf, shuffle):
    return DataLoader(CachedSet(idx, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, pin_memory=(device == "cuda"))

PRETRAINED = True
def build_model():
    weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if PRETRAINED else None
    m = models.mobilenet_v2(weights=weights)
    for p in m.features.parameters():
        p.requires_grad = False                     # แช่แข็งส่วนดึงฟีเจอร์
    m.classifier[1] = nn.Linear(m.last_channel, len(CLASSES))
    return m.to(device)

if USE_CLASS_WEIGHTS and not TEMP_MODE:
    w = json.load(open(os.path.join(PALM_OUT, "class_weights.json")))["weights"]
    class_weight = torch.tensor(w, dtype=torch.float32, device=device)
else:
    class_weight = None

def predict(model, idx):
    model.eval(); preds, probs = [], []
    with torch.no_grad():
        for x, _ in make_loader(idx, tf_plain, False):
            p = torch.softmax(model(x.to(device)), 1).cpu()
            probs.append(p); preds.append(p.argmax(1))
    return torch.cat(preds).numpy(), torch.cat(probs).numpy()

def scores(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y, p), "precision": pr, "recall": rc, "macro_f1": f1}

def train(train_idx, tf, epochs, val_idx=None, log=""):
    """เทรน 1 ครั้ง ถ้ามี val_idx จะเก็บ epoch ที่ macro F1 บน validation ดีที่สุด"""
    seed_all(SEED)
    model = build_model()
    opt = torch.optim.Adam(model.classifier.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss(weight=class_weight)
    loader = make_loader(train_idx, tf, True)
    best, best_state, best_ep, history = -1, None, epochs, []
    for ep in range(1, epochs + 1):
        model.train(); model.features.eval()        # ให้ส่วนที่แช่แข็งคงที่จริง
        total, loss_sum = 0, 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = crit(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y); total += len(y)
        row = {"epoch": ep, "train_loss": loss_sum / total}
        if val_idx is not None:
            p, _ = predict(model, val_idx)
            row.update(scores(labels[val_idx], p))
            if row["macro_f1"] > best:
                best, best_state, best_ep = row["macro_f1"], copy.deepcopy(model.state_dict()), ep
        history.append(row)
        print(f"{log} epoch {ep}: " + " | ".join(f"{k} {v:.3f}" for k, v in row.items() if k != "epoch"))
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, history

## ขั้นที่ 4: Cross validation เปรียบเทียบ 2 การทดลอง
แต่ละการทดลองเทรน 5 รอบ รอบที่ k ใช้ fold k เป็น validation และ fold อื่นเป็น train
ผลแต่ละรอบเซฟลง `cv_results.json` ทันที

In [ ]:
train_rows = split.index[split["split"] == "train"].to_numpy()
test_idx = split.index[split["split"] == "test"].to_numpy()
folds = split.loc[train_rows, "fold"].to_numpy()
run_folds = [0] if MODE == "quick" else sorted(set(folds))

res_file = os.path.join(OUT_DIR, f"cv_results_{MODE}.json")
results = json.load(open(res_file)) if os.path.exists(res_file) and not TEMP_MODE else []
done = {(r["experiment"], r["fold"]) for r in results}
quick_models = {}

for name, tf in EXPERIMENTS.items():
    for k in run_folds:
        if (name, k) in done and MODE == "cv":
            print(f"ข้าม {name} fold {k} (ทำแล้ว)"); continue
        t0 = time.time()
        tr_idx, va_idx = train_rows[folds != k], train_rows[folds == k]
        model, best_ep, hist = train(tr_idx, tf, EPOCHS, va_idx, log=f"[{name} fold {k}]")
        p, _ = predict(model, va_idx)
        r = {"experiment": name, "fold": int(k), "best_epoch": best_ep, **scores(labels[va_idx], p),
             "minutes": round((time.time() - t0) / 60, 1)}
        results = [x for x in results if not (x["experiment"] == name and x["fold"] == k)] + [r]
        if MODE == "quick":
            quick_models[name] = model
        if not TEMP_MODE:
            json.dump(results, open(res_file, "w"), indent=2)
        print(f"→ {name} fold {k}: macro F1 {r['macro_f1']:.3f} ({r['minutes']} นาที)\n")

In [ ]:
cv = pd.DataFrame(results)
metrics = ["accuracy", "precision", "recall", "macro_f1"]
table = cv.groupby("experiment")[metrics].agg(["mean", "std"]).round(3)
print("ผลบน validation (เฉลี่ย ± ส่วนเบี่ยงเบนมาตรฐาน ทุก fold):")
summary_cv = pd.DataFrame({m: [f"{table.loc[e, (m, 'mean')]:.3f} ± {0 if np.isnan(table.loc[e, (m, 'std')]) else table.loc[e, (m, 'std')]:.3f}"
                               for e in table.index] for m in metrics}, index=table.index)
display(summary_cv)
summary_cv.to_csv(os.path.join(OUT_DIR, f"cv_summary_{MODE}.csv"), encoding="utf-8-sig")

BEST = cv.groupby("experiment")["macro_f1"].mean().idxmax()
FINAL_EPOCHS = int(np.median(cv[cv.experiment == BEST]["best_epoch"]))
print(f"\nการทดลองที่ดีที่สุด: {BEST}")
if MODE == "cv":
    print(f"จะเทรนรอบสุดท้าย {FINAL_EPOCHS} epoch (ค่ากลางของ epoch ที่ดีที่สุดแต่ละ fold)")

fig, ax = plt.subplots(figsize=(6, 4))
for i, (name, g) in enumerate(cv.groupby("experiment")):
    ax.scatter(np.full(len(g), i) + np.random.uniform(-.05, .05, len(g)), g["macro_f1"], alpha=.6)
    ax.errorbar(i + .15, g["macro_f1"].mean(), yerr=g["macro_f1"].std() if len(g) > 1 else 0, fmt="o", color="black", capsize=6)
ax.set_xticks(range(cv["experiment"].nunique())); ax.set_xticklabels(sorted(cv["experiment"].unique()))
ax.set_ylabel("validation macro F1"); ax.set_title("Experiment comparison (each dot = one fold)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "01_cv_comparison.png"), dpi=150); plt.show()

## ขั้นที่ 5: เทรนโมเดลสุดท้าย
- **โหมด cv:** เทรนการทดลองที่ดีที่สุดใหม่กับ train ทั้งหมด (ทุก fold) ตามจำนวน epoch ที่ได้จากขั้นก่อน
- **โหมด quick:** ใช้โมเดลจาก fold 0 ของการทดลองที่ดีที่สุดเลย

In [ ]:
if MODE == "quick":
    final_model = quick_models[BEST]
else:
    final_model, _, _ = train(train_rows, EXPERIMENTS[BEST], FINAL_EPOCHS, log=f"[final {BEST}]")

## ขั้นที่ 6: วัดผลกับ test (ครั้งเดียว)
วัด 2 ระดับ:
- **ระดับคลาส:** ทายถูกไหมว่าเป็นอะไร (13 คลาส)
- **ระดับประเภทขยะ:** แปลงคำตอบด้วย `category_map` แล้วดูว่าทายถูกไหมว่าลงถังไหน

In [ ]:
test_pred, test_prob = predict(final_model, test_idx)
y_true = labels[test_idx]
cat_true, cat_pred = cls_to_cat[y_true], cls_to_cat[test_pred]

overall = pd.DataFrame([{"ระดับ": "คลาส (13)", **scores(y_true, test_pred)},
                        {"ระดับ": "ประเภทขยะ (5)", **scores(cat_true, cat_pred)}]).set_index("ระดับ").round(3)
display(overall)
overall.to_csv(os.path.join(OUT_DIR, "test_scores.csv"), encoding="utf-8-sig")

print("\nรายคลาส:")
print(classification_report(y_true, test_pred, labels=range(len(CLASSES)), target_names=CLASSES, zero_division=0))
print("รายประเภทขยะ:")
print(classification_report(cat_true, cat_pred, labels=range(len(CATEGORIES)), target_names=CATEGORIES, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(confusion_matrix(y_true, test_pred, labels=range(len(CLASSES))), display_labels=CLASSES).plot(
    ax=ax, cmap="Greens", xticks_rotation=60, colorbar=False)
ax.set_title(f"Confusion matrix - classes ({BEST})"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_confusion_classes.png"), dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(cat_true, cat_pred, labels=range(len(CATEGORIES))), display_labels=CATEGORIES).plot(
    ax=ax, cmap="Greens", xticks_rotation=30, colorbar=False)
ax.set_title("Confusion matrix - waste categories"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "03_confusion_categories.png"), dpi=150); plt.show()

cm = confusion_matrix(y_true, test_pred, labels=range(len(CLASSES)))
np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], CLASSES[i], CLASSES[j]) for i in range(len(CLASSES)) for j in range(len(CLASSES)) if cm[i, j]), reverse=True)[:8]
print("คู่ที่สับสนบ่อยที่สุด (จริง → ทาย):")
for n, a, b in pairs:
    print(f"  {a} → {b}: {n} ภาพ" + ("  ⚠️ ผิดประเภทขยะด้วย" if CATEGORY_MAP[a] != CATEGORY_MAP[b] else ""))

### วิเคราะห์ภาพที่ทายผิด
เลือกภาพที่ผิด**ประเภทขยะ**มาก่อน (ผิดแบบนี้ร้ายแรงกว่า เพราะบอกผู้ใช้ให้ทิ้งผิดถัง)

In [ ]:
wrong = np.where(test_pred != y_true)[0]
wrong = sorted(wrong, key=lambda i: (cat_true[i] == cat_pred[i], -test_prob[i].max()))[:12]
if wrong:
    fig, axes = plt.subplots(3, 4, figsize=(14, 11))
    for ax in axes.flat: ax.axis("off")
    for ax, i in zip(axes.flat, wrong):
        ax.imshow(images[test_idx[i]])
        bad_cat = cat_true[i] != cat_pred[i]
        ax.set_title(f"true: {CLASSES[y_true[i]]}\npred: {CLASSES[test_pred[i]]} ({test_prob[i].max():.0%})",
                     color="red" if bad_cat else "darkorange", fontsize=10)
    plt.suptitle("Misclassified test images (red = wrong waste category)"); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "04_misclassified.png"), dpi=150); plt.show()
print(f"ทายผิด {int((test_pred != y_true).sum())} จาก {len(y_true)} ภาพ "
      f"| ผิดประเภทขยะ {int((cat_pred != cat_true).sum())} ภาพ")

### เกณฑ์ "ระบบไม่แน่ใจ"
เว็บจะขึ้นว่า "ไม่แน่ใจ ลองถ่ายใหม่" เมื่อความมั่นใจต่ำกว่าเกณฑ์ ตารางนี้ช่วยเลือกเกณฑ์: ยิ่งเกณฑ์สูง คำตอบที่ระบบกล้าตอบจะแม่นขึ้น แต่จะตอบว่าไม่แน่ใจบ่อยขึ้น

In [ ]:
conf = test_prob.max(1)
rows = []
for t in [0.4, 0.5, 0.6, 0.7, 0.8]:
    m = conf >= t
    rows.append({"เกณฑ์": t, "ตอบได้ (%)": round(100 * m.mean(), 1),
                 "ความแม่นเมื่อตอบ (%)": round(100 * (test_pred[m] == y_true[m]).mean(), 1) if m.any() else None})
thr = pd.DataFrame(rows); display(thr)
thr.to_csv(os.path.join(OUT_DIR, "confidence_threshold.csv"), index=False, encoding="utf-8-sig")

## ขั้นที่ 7: Export โมเดลให้เว็บ
ไฟล์ `model.pt` เก็บน้ำหนักโมเดล รายชื่อคลาส ประเภทขยะ และค่าที่ใช้ปรับภาพ ส่งไฟล์นี้ให้ A

In [ ]:
export = {"arch": "mobilenet_v2", "state_dict": final_model.cpu().state_dict(), "classes": CLASSES,
          "category_map": CATEGORY_MAP, "img_size": IMG_SIZE, "mean": MEAN, "std": STD, "experiment": BEST}
torch.save(export, os.path.join(OUT_DIR, "model.pt"))
final_model.to(device)
print("บันทึก model.pt แล้ว", f"({os.path.getsize(os.path.join(OUT_DIR, 'model.pt')) / 1e6:.1f} MB)")
print("\nไฟล์ใน", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)): print(" -", f)

## ขั้นที่ 8: ลองทายรูปจริง
อัปโหลดรูปขยะที่ถ่ายเอง ใช้เช็กก่อนส่งโมเดลให้ทำเว็บ

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    uploaded = {}
for fn in uploaded:
    img = Image.open(fn).convert("RGB")
    final_model.eval()
    with torch.no_grad():
        prob = torch.softmax(final_model(tf_plain(img).unsqueeze(0).to(device)), 1)[0].cpu().numpy()
    top = prob.argsort()[::-1][:3]
    text = "\n".join(f"{CLASSES[i]} ({CATEGORY_MAP[CLASSES[i]]}): {prob[i]:.0%}" for i in top)
    plt.imshow(img); plt.axis("off"); plt.title(text); plt.show()

## เอาไปใช้ในสไลด์และรายงาน
- `cv_summary_cv.csv` + `01_cv_comparison.png` → เปรียบเทียบการทดลอง A กับ B
- `test_scores.csv` → ผลสุดท้าย 2 ระดับ
- `02_confusion_classes.png`, `03_confusion_categories.png`, `04_misclassified.png` → วิเคราะห์ข้อผิดพลาด
- `confidence_threshold.csv` → เลือกเกณฑ์ "ไม่แน่ใจ" ให้เว็บ

## เผื่ออาจารย์ถาม
- **ทำไมใช้ CNN แบบ transfer learning** MobileNetV2 เรียนรู้การดึงลักษณะภาพ (ขอบ พื้นผิว รูปทรง) จากภาพกว่าล้านภาพมาแล้ว เราจึงใช้ข้อมูลไม่กี่พันภาพเทรนแค่ชั้นสุดท้ายได้ เร็วและแม่นกว่าเริ่มจากศูนย์
- **ทำไมใช้ macro F1** ให้น้ำหนักทุกคลาสเท่ากัน ถ้าโมเดลทายคลาสเล็ก (เช่น trash_general) ผิดหมด คะแนนจะตกให้เห็น ขณะที่ accuracy รวมอาจยังดูสูง
- **ทำไมใช้ 5-fold** คะแนนจาก validation ชุดเดียวอาจฟลุ๊ค การหมุนให้ทุก fold ได้เป็น validation แล้วดูค่าเฉลี่ยและส่วนเบี่ยงเบน ช่วยบอกว่าการทดลองหนึ่งดีกว่าอีกอันจริงหรือไม่
- **ทำไมวัด 2 ระดับ** ทายผิดระหว่างแก้วสีขาวกับแก้วสีน้ำตาลยังลงถังรีไซเคิลถูก แต่ทายแบตเตอรี่เป็นโลหะคือบอกผู้ใช้ให้ทิ้งขยะอันตรายผิดถัง จึงต้องดูแยกกัน
- **ทำไมเทรนใหม่กับ train ทั้งหมดก่อนวัด test** หลังเลือกการทดลองได้แล้ว การใช้ข้อมูลทุก fold เทรนทำให้โมเดลได้เห็นข้อมูลมากที่สุด และ test ยังไม่เคยถูกใช้เลือกอะไรเลย